# 交互式问诊数据转换脚本

将 SMHC MiroDiag 数据转换为交互式问诊训练格式
- 不包含完整对话历史
- 只包含 Patient ID 和诊断结果
- Doctor Agent 将通过 Patient Agent API 进行实时交互


In [1]:
import json
import pandas as pd
import re
import os
from typing import Dict, List, Any, Tuple


In [2]:
def extract_major_icd_codes(full_icd_code: str) -> List[str]:
    """
    从完整 ICD 代码中提取大类代码（前3位）
    
    Args:
        full_icd_code: 完整的 ICD 代码，例如 'F20.400'
        
    Returns:
        大类代码列表，例如 ['F20']
    """
    if not full_icd_code:
        return []
    
    # 提取 F 开头的三位大类代码
    match = re.match(r'(F\d{2})', full_icd_code)
    if match:
        return [match.group(1)]
    return []


def create_interactive_diagnosis_prompt() -> List[Dict[str, str]]:
    """
    创建交互式问诊的 system prompt + 初始 user message
    
    Returns:
        包含 system prompt 和初始 user message 的消息列表
    """
    system_prompt = """
你是一名精神卫生中心的临床心理科医师，进行专业问诊。
遵循 ICD-10 精神与行为障碍标准，先通过充分的共情式问诊收集信息，再进行诊断。

## 核心问诊原则

1. **建立治疗性关系**：
   - 在提问时表达对患者困扰的理解和共情
   - 使用支持性语言，如"我能理解这一定让您很困扰"、"听起来您经历了很艰难的时期"
   - 避免冷漠的连续提问，在关键信息后给予情感回应

2. **深入追问症状细节**：
   - 对患者描述的每个症状进行具体化：持续时间、频率、严重程度、触发因素
   - 询问症状的时间进程：何时开始、是否加重、是否有缓解期
   - 探查症状对日常功能的影响：工作、人际关系、自我照顾能力
   - 使用开放式问题引导患者详细描述，如"能详细说说吗"、"还有其他表现吗"

3. **验证症状真实性**：
   - 对患者自述的症状进行交叉验证：询问具体事例、客观表现
   - 观察症状描述的一致性和逻辑性
   - 区分患者的主观感受与客观症状

4. **鉴别诊断**：
   - 系统性排查相似疾病的鉴别点
   - 针对性询问关键区分症状（如抑郁与双相的既往躁狂史、焦虑与惊恐的发作模式）
   - 探查器质性因素：用药史、躯体疾病、物质使用
   - 评估共病可能：多个诊断可能并存时需要分别评估

## ICD-10 精神与行为障碍标准大类

请仅从以下ICD-10标准中的10种疾病中选择最符合的诊断大类以及进一步细分的ICD-10小类：

- F32 抑郁发作：情绪持续低落、兴趣/愉快感下降、精力不足；伴睡眠/食欲改变、自责/无价值感等；可轻/中/重度（重度可伴精神病性症状）；无既往躁狂/轻躁狂。
  *鉴别要点：需排除双相障碍（F31）、适应障碍（F43）、器质性情绪障碍*

- F41 其他焦虑障碍：恐慌发作或广泛性焦虑为主；过度担忧、紧张、心悸、胸闷、出汗、眩晕、濒死感/失控感；与特定情境无关或不成比例，造成显著痛苦/功能损害。
  *鉴别要点：需排除躯体疾病（如甲亢）、物质诱发、抑郁伴焦虑*

- F39.9 未特指的心境（情感）障碍：存在心境障碍证据，但资料不足以明确归入抑郁或双相等具体亚型时选用。

- F51 非器质性睡眠障碍：失眠、过度嗜睡、梦魇、昼夜节律紊乱等；非器质性原因；睡眠问题为主要主诉并致显著困扰/功能损害。
  *鉴别要点：需排除抑郁/焦虑继发失眠、睡眠呼吸暂停等器质性原因*

- F98 其他儿童和青少年行为与情绪障碍：多见于儿童期起病（如遗尿/遗粪、口吃、抽动相关习惯性问题等），以发育期特异表现为主。

- F42 强迫障碍：反复的强迫观念/行为，个体自知过度或不合理但难以抵抗，耗时或致显著困扰/损害。
  *鉴别要点：需排除精神分裂症的思维障碍、抑郁的反刍思维*

- F31 双相情感障碍：既往或目前存在躁狂/轻躁狂发作与抑郁发作的交替或混合；需有明确躁狂谱系证据。
  *鉴别要点：详细询问既往是否有情绪高涨、精力旺盛、睡眠需求减少、冲动行为等躁狂症状*

- F43 对严重应激反应和适应障碍：与明确应激事件有关；可为急性应激反应、PTSD或适应障碍；核心包含再体验、回避、警觉性增高或与应激源相关的情绪/行为改变。
  *鉴别要点：需确认应激源存在、症状与应激源的时间关联、是否超出正常应激反应*

- F45 躯体形式障碍：反复或多样躯体症状为主（如疼痛、心悸、胃肠不适等），检查难以找到足以解释的器质性原因或与病因不相称，显著痛苦/就诊反复。
  *鉴别要点：需详细了解躯体检查结果、症状与情绪的关联、疾病信念*

- F20 精神分裂症：在知觉、思维、情感及行为等方面的广泛障碍；常见持续性妄想、幻听、思维松弛/破裂、情感淡漠、阴性症状，病程≥1月（或依本地标准）。
  *鉴别要点：需评估现实检验能力、思维连贯性、情感协调性*

- Z71 咨询和医疗建议相关因素：包括心理咨询、健康教育、生活方式指导等，当患者主要需要咨询服务而非特定疾病治疗时使用。

## 问诊流程建议

1. **开场阶段**（1-2轮）：表达欢迎，了解主诉，建立初步信任
2. **症状探查阶段**（8-15轮）：深入了解核心症状及伴随症状，充分细节追问
3. **鉴别诊断阶段**（3-5轮）：针对性询问鉴别点，排除其他可能诊断
4. **功能评估阶段**（2-3轮）：评估症状对生活功能的影响
5. **诊断决策阶段**：综合信息，给出诊断

## 输出规范（只选其一）

1) **问诊阶段**：<think>思考过程</think> [共情回应+]问句
   - 例如："<think>患者提到情绪低落，需要了解持续时间和严重程度</think> 我能理解您现在很不舒服。这种情绪低落的感觉大概持续多久了？每天都会这样吗？"

2) **诊断阶段**：<think>诊断推理过程（需包含症状总结、鉴别诊断分析）</think> <box>ICD-10编码</box>
   - ICD-10小类可多项，用分号分隔，如 <box>F32.1; F41.2</box>
   - 诊断思考必须包含：①主要症状总结 ②鉴别诊断排除 ③诊断依据

## 重要提示

- 当信息不足以排除其他疾病进行诊断时，**必须**继续调用工具 ask_patient 进行深入问诊
- 避免过早下诊断，确保收集到充分的鉴别信息
- 当确信可以诊断时，调用工具 do_diagnose，并在参数 diagnosis 中填入完整诊断文本
- 保持温暖、专业的医患沟通风格，在专业性与人文关怀之间取得平衡
"""
    
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "你好医生。"
        }
    ]


In [3]:

def make_interactive_map_fn(data_source: str, split: str):
    """
    创建交互式问诊数据转换函数
    
    Args:
        data_source: 数据源标识
        split: 'train' 或 'val'
    
    Returns:
        数据转换函数
    """
    def process_fn(data_item: Dict[str, Any], idx: int) -> Dict[str, Any]:
        """
        处理单个数据项
        
        Args:
            data_item: 原始数据项
            idx: 索引
            
        Returns:
            转换后的数据项，如果数据无效则返回 None
        """
        # 提取 patient_id 和诊断代码
        patient_id = data_item.get('patient_id')
        full_icd_code = data_item.get('DiagnosisCode', '')
        
        # 过滤无效数据
        if not patient_id or not full_icd_code:
            return None
        
        # 提取大类 ICD 代码
        major_icd_codes = extract_major_icd_codes(full_icd_code)
        
        if not major_icd_codes:
            return None
        
        # 创建 prompt（只包含 system prompt，不包含对话历史）
        prompt = create_interactive_diagnosis_prompt()
        
        # 构建转换后的数据
        converted_data = {
            "data_source": data_source,
            "prompt": prompt,
            "raw_prompt": prompt,
            "ability": "interactive_diagnosis",  # 交互式问诊能力
            "reward_model": {
                "style": "rule",
                "ground_truth": major_icd_codes
            },
            "extra_info": {
                "split": split,
                "index": idx,
                "visit_number": patient_id,  # 用 patient_id 作为 visit_number
                "full_icd_code": [full_icd_code],
                "diagnosis_name": data_item.get('Diagnosis', ''),
                "patient_id": patient_id,  # 添加 patient_id 到 extra_info
                "tools_kwargs": {
                    "ask_patient": {
                        "create_kwargs": {
                            "patient_id": patient_id,
                            "patient_version": "v1",
                            "model_name": "moonshotai/kimi-k2-0905"
                        }
                    },
                    "do_diagnose": {
                        "create_kwargs": {
                            "patient_id": patient_id,
                            "ground_truth": major_icd_codes,
                            "patient_version": "v1",
                            "model_name": "moonshotai/kimi-k2-0905"
                        }
                    }
                }
            }
        }
        
        return converted_data
    
    return process_fn


In [4]:
def convert_to_interactive_diagnosis_parquet(
    train_file: str,
    output_dir: str,
    train_ratio: float = 0.9,
    max_samples: int = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    将数据转换为交互式问诊训练格式的 parquet 文件
    
    Args:
        train_file: 训练数据文件路径
        output_dir: 输出目录
        train_ratio: 训练集比例（默认 0.9）
        max_samples: 最大样本数（用于测试）
        
    Returns:
        (train_df, val_df) 训练集和验证集 DataFrame
    """
    data_source = "SMHC_RL_interactive_diagnosis"
    
    print(f"正在读取数据: {train_file}")
    with open(train_file, 'r', encoding='utf-8') as f:
        all_data = json.load(f)
    
    print(f"  读取到 {len(all_data)} 条数据")
    
    # 限制样本数（用于测试）
    if max_samples:
        all_data = all_data[:max_samples]
        print(f"  限制样本数为 {max_samples}")
    
    # 去重：基于 patient_id
    print("\\n正在按 patient_id 去重...")
    seen_ids = set()
    deduplicated_data = []
    
    for item in all_data:
        patient_id = item.get('patient_id', '')
        if patient_id and patient_id not in seen_ids:
            seen_ids.add(patient_id)
            deduplicated_data.append(item)
    
    print(f"去重后: {len(all_data)} -> {len(deduplicated_data)} 条")
    
    # 按比例划分训练集和验证集
    split_idx = int(len(deduplicated_data) * train_ratio)
    train_data = deduplicated_data[:split_idx]
    val_data = deduplicated_data[split_idx:]
    
    print(f"\\n数据划分:")
    print(f"  训练集: {len(train_data)} 条")
    print(f"  验证集: {len(val_data)} 条")
    
    # 创建转换函数
    train_map_fn = make_interactive_map_fn(data_source, 'train')
    val_map_fn = make_interactive_map_fn(data_source, 'val')
    
    # 转换训练数据
    print("\\n正在转换训练数据...")
    converted_train = []
    filtered_train_count = 0
    
    for idx, item in enumerate(train_data):
        result = train_map_fn(item, idx)
        if result is not None:
            converted_train.append(result)
        else:
            filtered_train_count += 1
    
    # 转换验证数据
    print("正在转换验证数据...")
    converted_val = []
    filtered_val_count = 0
    
    for idx, item in enumerate(val_data):
        result = val_map_fn(item, idx)
        if result is not None:
            converted_val.append(result)
        else:
            filtered_val_count += 1
    
    # 输出过滤统计
    print(f"\\n过滤统计:")
    print(f"  训练集: 原始 {len(train_data)} 条，过滤 {filtered_train_count} 条，保留 {len(converted_train)} 条")
    print(f"  验证集: 原始 {len(val_data)} 条，过滤 {filtered_val_count} 条，保留 {len(converted_val)} 条")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 转换为 DataFrame 并保存为 parquet
    print("\\n正在保存为 parquet 格式...")
    train_df = pd.DataFrame(converted_train)
    val_df = pd.DataFrame(converted_val)
    
    train_output_path = os.path.join(output_dir, 'train.parquet')
    val_output_path = os.path.join(output_dir, 'val.parquet')
    
    train_df.to_parquet(train_output_path, index=False)
    val_df.to_parquet(val_output_path, index=False)
    
    print(f"\\n训练数据已保存至: {train_output_path}")
    print(f"验证数据已保存至: {val_output_path}")
    print(f"训练数据量: {len(converted_train)} 条")
    print(f"验证数据量: {len(converted_val)} 条")
    
    # 显示样例数据
    if len(converted_train) > 0:
        print("\\n样例数据 (训练集第一条):")
        print(json.dumps(converted_train[0], ensure_ascii=False, indent=2))
    
    return train_df, val_df


In [5]:
# 配置参数
TRAIN_FILE = "/tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json"
OUTPUT_DIR = "/tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy"
TRAIN_RATIO = 0.9
MAX_SAMPLES = None  # 设置为 None 使用全部数据，或设置为数字进行测试（如 100）

# 执行转换
print("开始执行交互式问诊数据转换...")
print("="*80)
print(f"输入文件: {TRAIN_FILE}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"训练集比例: {TRAIN_RATIO}")
if MAX_SAMPLES:
    print(f"限制样本数: {MAX_SAMPLES}")
print("="*80)

train_df, val_df = convert_to_interactive_diagnosis_parquet(
    train_file=TRAIN_FILE,
    output_dir=OUTPUT_DIR,
    train_ratio=TRAIN_RATIO,
    max_samples=MAX_SAMPLES
)

print("\\n" + "="*80)
print("转换完成！")
print("="*80)


开始执行交互式问诊数据转换...
输入文件: /tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json
输出目录: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy
训练集比例: 0.9
正在读取数据: /tcci_mnt/shihao/project/Lingxi_annotation_0111/raw_data/LingxiDiag-16K_train_data.json
  读取到 14000 条数据
\n正在按 patient_id 去重...
去重后: 14000 -> 13997 条
\n数据划分:
  训练集: 12597 条
  验证集: 1400 条
\n正在转换训练数据...
正在转换验证数据...
\n过滤统计:
  训练集: 原始 12597 条，过滤 371 条，保留 12226 条
  验证集: 原始 1400 条，过滤 43 条，保留 1357 条
\n正在保存为 parquet 格式...
\n训练数据已保存至: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy/train.parquet
验证数据已保存至: /tcci_mnt/shihao/project/verl/psy_r1/dataset_rl/LingxiDiag-interactive-long-empathy/val.parquet
训练数据量: 12226 条
验证数据量: 1357 条
\n样例数据 (训练集第一条):
{
  "data_source": "SMHC_RL_interactive_diagnosis",
  "prompt": [
    {
      "role": "system",
      "content": "\n你是一名精神卫生中心的临床心理科医师，进行专业问诊。\n遵循 ICD-10 精神与行为障碍标准，先通过充分的共情式问诊收集信息，再进行诊断。\n\n## 核心问诊原则\n\n

In [6]:
# 验证生成的数据
print("\\n数据验证:")
print("="*80)

print(f"\\n训练集数据量: {len(train_df)}")
print(f"验证集数据量: {len(val_df)}")

print("\\n数据列名:")
print(train_df.columns.tolist())

# 检查数据结构
print("\\n训练集第一条数据的结构:")
sample = train_df.iloc[0]
print(f"- data_source: {sample['data_source']}")
print(f"- ability: {sample['ability']}")
print(f"- prompt: 包含 {len(sample['prompt'])} 个消息")
print(f"- reward_model: {sample['reward_model']}")
print(f"- extra_info keys: {list(sample['extra_info'].keys())}")

# 显示 system prompt
print("\\nSystem Prompt 内容:")
print("-"*80)
print(sample['prompt'][0]['content'][:500] + "...")  # 只显示前500字符
print("-"*80)

# 统计诊断代码分布
print("\\n诊断代码分布:")
train_diagnosis = train_df['reward_model'].apply(lambda x: str(x['ground_truth']))
print("\\n训练集 (前 20 个):")
print(train_diagnosis.value_counts().head(20))

val_diagnosis = val_df['reward_model'].apply(lambda x: str(x['ground_truth']))
print("\\n验证集 (前 20 个):")
print(val_diagnosis.value_counts().head(20))

# 数据质量检查
print("\\n数据质量检查:")
print(f"训练集空值检查: {train_df.isnull().sum().sum()}")
print(f"验证集空值检查: {val_df.isnull().sum().sum()}")

print("\\n" + "="*80)
print("验证完成！数据已准备好用于交互式问诊训练")
print("="*80)


\n数据验证:
\n训练集数据量: 12226
验证集数据量: 1357
\n数据列名:
['data_source', 'prompt', 'raw_prompt', 'ability', 'reward_model', 'extra_info']
\n训练集第一条数据的结构:
- data_source: SMHC_RL_interactive_diagnosis
- ability: interactive_diagnosis
- prompt: 包含 2 个消息
- reward_model: {'style': 'rule', 'ground_truth': ['F41']}
- extra_info keys: ['split', 'index', 'visit_number', 'full_icd_code', 'diagnosis_name', 'patient_id', 'tools_kwargs']
\nSystem Prompt 内容:
--------------------------------------------------------------------------------

你是一名精神卫生中心的临床心理科医师，进行专业问诊。
遵循 ICD-10 精神与行为障碍标准，先通过充分的共情式问诊收集信息，再进行诊断。

## 核心问诊原则

1. **建立治疗性关系**：
   - 在提问时表达对患者困扰的理解和共情
   - 使用支持性语言，如"我能理解这一定让您很困扰"、"听起来您经历了很艰难的时期"
   - 避免冷漠的连续提问，在关键信息后给予情感回应

2. **深入追问症状细节**：
   - 对患者描述的每个症状进行具体化：持续时间、频率、严重程度、触发因素
   - 询问症状的时间进程：何时开始、是否加重、是否有缓解期
   - 探查症状对日常功能的影响：工作、人际关系、自我照顾能力
   - 使用开放式问题引导患者详细描述，如"能详细说说吗"、"还有其他表现吗"

3. **验证症状真实性**：
   - 对患者自述的症状进行交叉验证：询问具体事例、客观表现
   - 观察症状描述的一致性和逻辑性
   - 区分患者的主观感受与客观症状

4. **鉴别诊断**：
   - 系统性排查相似疾病的鉴别点
   